# Human-in-the-Loop Demo

This notebook shows a tiny LangGraph workflow that pauses for human review before finishing.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

In [ ]:
class ReviewState(TypedDict, total=False):
    topic: str
    draft: str
    approved: bool
    human_feedback: str
    final_message: str


def create_draft(state: ReviewState):
    topic = state["topic"]
    draft = (
        f"Draft announcement about {topic}: "
        "We are preparing a small update and want your review before it goes live."
    )
    return {"draft": draft, "approved": False, "human_feedback": ""}


def human_review(state: ReviewState):
    review = interrupt(
        {
            "question": "Approve this draft?",
            "draft": state["draft"],
            "resume_examples": [
                {"approved": True, "feedback": "Looks good."},
                {"approved": False, "feedback": "Make the tone more friendly."},
            ],
        }
    )

    return {
        "approved": review.get("approved", False),
        "human_feedback": review.get("feedback", ""),
    }


def finalize(state: ReviewState):
    if state["approved"]:
        return {"final_message": f"Published: {state['draft']}"}

    return {
        "final_message": (
            "Not published. Human feedback: "
            f"{state.get('human_feedback', 'No feedback provided.')}"
        )
    }

In [ ]:
builder = StateGraph(ReviewState)

builder.add_node("create_draft", create_draft)
builder.add_node("human_review", human_review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "create_draft")
builder.add_edge("create_draft", "human_review")
builder.add_edge("human_review", "finalize")
builder.add_edge("finalize", END)

checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

app

Run the graph once. It pauses at `interrupt()` and returns an interrupt payload instead of finishing.

In [ ]:
config = {"configurable": {"thread_id": "hitl-demo-1"}}

first = app.invoke(
    {"topic": "LangGraph human review"},
    config=config,
)

first

In [ ]:
# The review request sent to the human.
first["__interrupt__"][0].value

Resume the same thread with the human's decision. The value passed to `Command(resume=...)` becomes the return value of `interrupt()` inside `human_review`.

In [ ]:
final = app.invoke(
    Command(resume={"approved": True, "feedback": "Looks good."}),
    config=config,
)

final["final_message"]

Try a rejection on a fresh thread.

In [ ]:
reject_config = {"configurable": {"thread_id": "hitl-demo-2"}}

app.invoke({"topic": "A new chatbot feature"}, config=reject_config)

rejected = app.invoke(
    Command(resume={"approved": False, "feedback": "Needs a warmer tone."}),
    config=reject_config,
)

rejected["final_message"]